# RAR v2 — Pre-Registered Two-Task Campaign (Ollama + gemma2:9b on free GPU)

Runs the **pre-registered extended evaluation** (see `PREREGISTRATION.md`): N=10 seeds × 60 cycles, three conditions, with Phase-2 instrumentation (best-found + context-rot signal).

**Tasks (one kernel run per task — set `TASK` in cell 6):**
- `TASK='A'` — synthetic 10-class manifold (Phase-0 validated, spread ~35pp)
- `TASK='B'` — CIFAR-10 → leak-free PCA-64 (owner-verified spread 24.5pp)

## ⚙️ Before you run — Settings (right sidebar):
1. **Accelerator → GPU T4 x2** (or any GPU)
2. **Internet → On** (Ollama install, model pull, repo clone, CIFAR download)

Then **Save Version → Save & Run All**. ≈ 5–7 h per task. Each completed seed checkpoints to `partial_results.json`; re-running resumes. Output lands as `/kaggle/working/pilot_results_task{A|B}.json`.

## 1. GPU check

In [ ]:
import subprocess, shutil, torch
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], capture_output=True, text=True).stdout)
else:
    print('No nvidia-smi -> CPU-only instance (GPU quota may be exhausted).')
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
# NOTE: no assert -- the harness auto-detects GPU/CPU and adapts (CPU uses fewer epochs).

## 2. Install Ollama and start the server

In [ ]:
import os, time, subprocess, urllib.request, shutil
# Ollama's installer extracts a .tar.zst; Kaggle's base image lacks zstd, so install it first.
os.system('apt-get -qq update >/dev/null 2>&1 && apt-get -qq install -y zstd >/dev/null 2>&1')
rc = os.system('curl -fsSL https://ollama.com/install.sh | sh')
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
assert shutil.which('ollama') or os.path.exists(ollama_bin), f'Ollama install failed (rc={rc}).'
print('ollama binary:', ollama_bin)
# Start the server in the background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
server = subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# Wait for it to come up
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3); print('Ollama is up.'); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama server did not start.')

## 3. Pull the orchestrator model (Nemotron-Nano-9B GGUF, with fallback)

In [ ]:
import os
# NOTE: Nemotron-Nano-9B-v2 (Mamba-Transformer hybrid) does NOT GPU-accelerate on
# Kaggle's llama.cpp build (>5 min per short call -> not viable for a multi-call
# campaign; verified). We use gemma2:9b, a vetted open 9B instruct model, as the
# orchestrator. This is an honest substitution; the paper's model name must reflect it.
assert os.system(f'{ollama_bin} pull gemma2:9b') == 0, 'gemma2:9b pull failed'
MODEL = 'gemma2:9b'
print('USING MODEL:', MODEL)

## 4. Install Python deps and clone the repo

In [ ]:
import os
os.system('pip -q install aiohttp networkx scikit-learn scipy matplotlib nest_asyncio >/dev/null 2>&1')
# Fresh clone of the latest code
os.system('rm -rf /kaggle/working/rar && git clone --depth 1 https://github.com/Tahir-yamin/recursive-autonomy-research /kaggle/working/rar')
print(os.listdir('/kaggle/working/rar'))

## 5. Configure the local backend and health-check (one real JSON call)

In [ ]:
import os, sys, importlib, asyncio, nest_asyncio
nest_asyncio.apply()  # allow asyncio.run inside the Jupyter event loop
os.environ['LLM_BASE_URL'] = 'http://127.0.0.1:11434/v1/chat/completions'
os.environ['LLM_API_KEY'] = 'ollama'
os.environ['OPENROUTER_MODEL'] = MODEL
os.environ['OPENROUTER_MAX_TOKENS'] = '4000'
# Let the harness AUTO-DETECT a usable GPU (it probes a real CUDA op and falls back to
# CPU only if the GPU arch is unsupported, e.g. a Pascal P100). GPU is essential here:
# the v2 search includes 7-layer x 512-wide MLPs that are ~15x slower on CPU.
os.environ.pop('RAR_TORCH_DEVICE', None)
os.chdir('/kaggle/working/rar')
sys.path.insert(0, '/kaggle/working/rar')
import run_pilot_experiment as rp; importlib.reload(rp)
import run_deep_learning_harness as h
print('Training device:', h._select_device())

async def health():
    prompt = rp.SEARCH_SPACE_PROMPT + '\nPropose a config. Respond ONLY JSON. This is the first trial.'
    r = await rp.call_llm(prompt)
    cfg = rp.parse_json_response(r) if r else None
    print('RAW:', repr(r)[:160]); print('PARSED:', cfg); print('VALID:', rp.is_valid_config(cfg) if cfg else False)
    return rp.is_valid_config(cfg) if cfg else False
ok = asyncio.run(health())
assert ok, 'Health check failed: model did not return a valid JSON config. Try the fallback model.'

## 6. Run the full N=10 campaign on GPU
Real LLM proposals (local) + real PyTorch training (T4). Checkpoints each completed seed to `partial_results.json`. Re-running this cell resumes from the last completed seed.

In [ ]:
import os, asyncio, importlib, nest_asyncio, run_pilot_experiment as rp
nest_asyncio.apply()

# ====== PRE-REGISTERED CAMPAIGN CONFIG (PREREGISTRATION.md) ======
TASK = 'B'   # <<< SET 'A' (synthetic manifold) or 'B' (CIFAR-10->PCA64); one kernel run each
# =================================================================
os.environ['TASK'] = TASK
os.environ['RAR_CYCLES'] = '60'
os.environ['RAR_SEEDS'] = '101,107,113,127'  # default N=10 pre-registered seeds
importlib.reload(rp)
asyncio.run(rp.execute_campaign())
print(f'CAMPAIGN COMPLETE (TASK={TASK})')

## 7. Results summary + package for download

In [ ]:
import json, shutil, os, numpy as np
res = json.load(open('/kaggle/working/rar/pilot_results.json'))
print('TASK:', os.environ.get('TASK'), '| SEEDS:', res['SEEDS'])
print('Wilcoxon p (RAR vs Baseline):', res['wilcoxon_p_value_RAR_vs_Baseline'])
for c in ['stateless_baseline','vector_rag','rar_compressed']:
    cd = res['data']['conditions'][c]
    ta = np.array(cd['test_accuracies'])*100
    bf = cd.get('best_found_trajectories', [])
    final_bf = np.array([t[-1] for t in bf])*100 if bf else None
    ctx = cd.get('best_in_context_trajectories', [])
    late_vis = np.mean([np.mean(t[-10:]) for t in ctx])*100 if ctx else float('nan')
    llm = sum(cd.get('llm_proposal_counts', [])); heu = sum(cd.get('heuristic_proposal_counts', []))
    extra = f"  best-found(final)={final_bf.mean():.2f}%" if final_bf is not None else ""
    print(f"{c:18s} test={ta.mean():.2f}±{ta.std():.2f}%{extra}  late-cycle best-visible={late_vis:.0f}%  LLM={llm} heur={heu}")

# Central Phase-2 figure
os.system('cd /kaggle/working/rar && python plot_best_found.py')

# Package outputs per task (so A and B don't overwrite)
T = os.environ.get('TASK', 'A')
for src, dst in [('pilot_results.json', f'pilot_results_task{T}.json'),
                 ('partial_results.json', f'partial_results_task{T}.json'),
                 ('fig_best_found_vs_cycle.png', f'fig_best_found_vs_cycle_task{T}.png')]:
    p = f'/kaggle/working/rar/{src}'
    if os.path.exists(p): shutil.copy(p, f'/kaggle/working/{dst}')
print(f'\nDownload pilot_results_task{T}.json (+ figure) from the Output panel and send back.')